In [ ]:
!date

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import glob
import os
import re

from statsmodels.stats.multitest import multipletests

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2/multiome'

donors = sorted(list(np.loadtxt(f'{projdir}/txt/donors.txt', dtype=str)))
tmp_order = ['Start', 'Sendai', 'Delayed', 'Fail1', 'Inter1',  'Inter2', 'Fail2', 'IPS']
clusts = [f'{x}_{y}' for x,y in zip(np.repeat(tmp_order, len(donors)), np.tile(donors, len(tmp_order)))]

clusters = np.loadtxt(f'{projdir}/txt/clusters/clusters.txt', dtype=str)

In [ ]:
indir = f'{projdir}/motifs/peaks/cluster'
s = clusters[0]
motifs = pd.read_csv(f'{indir}/{s}/knownResults.txt', sep='\t', index_col=0).index
motifs = motifs[~motifs.duplicated()]
motifs.shape

### adapted from donor_leiden notebook

In [ ]:
motifs_renamed = pd.Index([x.split('(')[0].upper() for x in motifs])

###
tf_path = '/u/project/cluo/terencew/reference/motifs/human/TFClass_human_table_simple_SOX_collapsed.txt'
tf_meta = pd.read_csv(tf_path, sep='\t', header=None)
tf_dict = dict(zip(tf_meta[0], tf_meta[1]))

###
motifs_subfamily = np.array([tf_dict.get(x) for x in motifs_renamed])
match_mask = np.array([x != None for x in motifs_subfamily])
matched_motifs = motifs_subfamily[match_mask]
###
motif_families = np.unique(matched_motifs)
motif_families.shape

In [ ]:
tmp_clusters = ['Start', 'Fail1']

In [ ]:
epsilon = 1e-5
indir = f'{projdir}/motifs/peaks/start_fail1/other_bg'
homer_fc = pd.DataFrame(index=motifs)

for s in tmp_clusters:
    homer = pd.read_csv(f'{indir}/{s}/knownResults.txt', sep='\t', index_col=0)
    homer = homer[~homer.index.duplicated()]
    homer['target_pct'] = [float(x.split('%')[0]) for x in homer['% of Target Sequences with Motif']]
    homer['background_pct'] = [float(x.split('%')[0]) for x in homer['% of Background Sequences with Motif']]
    homer['target_pct'] += epsilon
    homer['background_pct'] += epsilon
    homer['fold_change'] = homer['target_pct'] / homer['background_pct']
    homer['log2_fold_change'] = np.log2(homer['fold_change'])
    homer_fc[s] = homer['log2_fold_change']

homer_fc = homer_fc.reindex(motifs)
homer_fc.index = motifs_subfamily
homer_fc = homer_fc[match_mask]
homer_fc.shape

In [ ]:
homer_fc.head()

In [ ]:
epsilon = 1e-5
indir = f'{projdir}/motifs/peaks/start_fail1/other_bg'
homer_pvals = pd.DataFrame(index=motifs)

for s in tmp_clusters:
    homer = pd.read_csv(f'{indir}/{s}/knownResults.txt', sep='\t', index_col=0)
    homer = homer[~homer.index.duplicated()]
    homer_pvals[s] = homer['P-value']

homer_pvals = homer_pvals.reindex(motifs)
homer_pvals.index = motifs_subfamily
homer_pvals = homer_pvals[match_mask]
homer_pvals.shape

In [ ]:
homer_pvals.head()

In [ ]:
whitelist = ['Start', 'Fail1']
mask = np.array([x.split('_')[0] in whitelist for x in homer_fc.columns])
tmp_homer_fc = homer_fc.loc[:,mask]
tmp_homer_pvals = homer_pvals.loc[:,mask]

In [ ]:
pvals_list = []
fc_list = []
for m in motif_families:
    tmp_pvals = tmp_homer_pvals[tmp_homer_pvals.index == m].reset_index(drop=True)
    tmp_fc = tmp_homer_fc[tmp_homer_fc.index == m].reset_index(drop=True)
    min_pval = tmp_pvals.min().values
    idx = tmp_pvals.idxmin(axis=0)
    min_fc = tmp_fc.lookup(idx, tmp_fc.columns)
    pvals_list.append(min_pval)
    fc_list.append(min_fc)

In [ ]:
tmp_homer_pvals_sub = pd.DataFrame(np.vstack(pvals_list))
tmp_homer_fc_sub = pd.DataFrame(np.vstack(fc_list))
tmp_homer_fc_sub.columns = tmp_homer_pvals.columns
tmp_homer_pvals_sub.columns = tmp_homer_pvals.columns
tmp_homer_fc_sub.index = motif_families
tmp_homer_pvals_sub.index = motif_families
tmp_homer_fc_sub.shape, tmp_homer_pvals_sub.shape

In [ ]:
tmp_homer_fdr_sub = pd.DataFrame(index=tmp_homer_pvals_sub.index, columns=tmp_homer_pvals_sub.columns)
for clust in tmp_homer_pvals_sub.columns:
    _, fdrs, _, _ = multipletests(tmp_homer_pvals_sub[clust], method='fdr_bh')
    tmp_homer_fdr_sub[clust] = fdrs
tmp_homer_fdr_sub = tmp_homer_fdr_sub.astype(float)
tmp_homer_fdr_sub.shape

In [ ]:
thresh = 1e-5
mask = tmp_homer_fdr_sub < thresh
tmp_mask = mask.sum(axis=1) > 0

tmp_homer_fc_sub_filt = tmp_homer_fc_sub.loc[tmp_mask,:].T
tmp_homer_fdr_sub_filt = tmp_homer_fdr_sub.loc[tmp_mask,:].T
tmp_homer_fc_sub_filt.shape, tmp_homer_fdr_sub_filt.shape

In [ ]:
tmp_homer_fc_sub_filt.head()

In [ ]:
tmp_dict = { 'Factors with 2-3 adjacent zinc fingersand a BTB/POZ domain' : 'Zinc/BTB',
           'FTZ-F1-related receptors (NR5)' : 'FTZ-F1',
           'PAX-2-like (partial homeobox)' : 'PAX-2-like',
            'Retinoic acid receptors (NR1B)' :  'NR1B',
             'Thyroid hormone receptors (NR1A)' : ' NR1A'
           }
tmp_homer_fc_sub_filt.columns = [tmp_dict.get(x,x) for x in tmp_homer_fc_sub_filt.columns]
tmp_homer_fdr_sub_filt.columns = [tmp_dict.get(x,x) for x in tmp_homer_fdr_sub_filt.columns]

In [ ]:
tmp_homer_fc_sub.max(axis=0), \
tmp_homer_fc_sub.min(axis=0)

In [ ]:
sns.clustermap(tmp_homer_fc_sub_filt, cmap="coolwarm",
                   method="average", 
                   row_cluster=True, col_cluster=True, vmin=-1.0, vmax=2.5, figsize=(12,6))

In [ ]:
outdir = '/u/project/cluo/terencew/igvf/2023_YR2/final_figures/csv/figure3'
tmp_homer_fc_sub_filt.to_csv(f'{outdir}/start_fail1_homer_fc.csv', sep='\t')
tmp_homer_fdr_sub_filt.to_csv(f'{outdir}/start_fail1_homer_pvals.csv', sep='\t')

In [ ]:
!date